# Flatten KV (int4): fused K/V Hadamard vs reference

Sanity-checks `fuse_k_hadamard` and `fuse_v_hadamard` against **flatten without fuse** plus **pure PyTorch FWHT** on K and/or V (same reshape and `1/sqrt(order)` as FlashAttention prefill for Q).

Benchmarks throughput: baseline, K-only fuse, V-only fuse, and K+V fuse.

**Requirements:** CUDA, `triton`, `sglang` on `PYTHONPATH` (e.g. `pip install -e ./python` from this repo). **Int4 only** — either fuse flag raises for int8.

**Byte model (per `flatten_kv_cache_sglang` call):**
- **Read:** packed K `T_slot×H×(D/2)` uint8 + same for V + K/V scales `T_slot×H×2×4` fp32 + page table (batch×pages) int32 + small metadata.
- **Write:** flattened K and V `total_tokens×H×D×elem` in `output_dtype`.

Use the same formula for fused vs unfused so GB/s comparison reflects the extra compute.

In [1]:
from __future__ import annotations

import math
import os
import sys
import time

import torch

REPO_ROOT = os.environ.get("SGLANG_REPO_ROOT", os.getcwd())
PY_ROOT = os.path.join(REPO_ROOT, "python")
if os.path.isdir(PY_ROOT) and PY_ROOT not in sys.path:
    sys.path.insert(0, PY_ROOT)

assert torch.cuda.is_available(), "CUDA required"
DEVICE = torch.device("cuda")
torch.manual_seed(0)

from sglang.srt.mem_cache.kv_quant_kernels import quantized_set_kv_int4_triton
from sglang.srt.layers.attention.kernels.flatten_kv_cache import flatten_kv_cache_sglang

## Reference: segmented FWHT on last dim (matches decode / FA Q path)

No `fast_hadamard_transform` dependency — iterative butterfly in PyTorch float32.

In [2]:
def fwht_last_dim_1d(v: torch.Tensor) -> torch.Tensor:
    """FWHT on last dimension; v.shape[-1] must be a power of two."""
    n = v.shape[-1]
    x = v.clone()
    h = 1
    while h < n:
        view = x.reshape(*x.shape[:-1], -1, 2 * h)
        a = view[..., :h]
        b = view[..., h : 2 * h]
        new_a = a + b
        new_b = a - b
        x = torch.cat([new_a, new_b], dim=-1).reshape_as(x)
        h *= 2
    return x


def ref_head_hadamard_after_flatten(x: torch.Tensor, order: int) -> torch.Tensor:
    """Same as FA prefill: view(..., D//order, order), scale 1/sqrt(order), FWHT on last dim."""
    d_last = x.shape[-1]
    lead = x.shape[:-1]
    y = x.reshape(-1, d_last // order, order).float()
    y = y * (1.0 / math.sqrt(float(order)))
    y = fwht_last_dim_1d(y)
    return y.reshape(*lead, d_last).to(x.dtype)


# Alias for readability in K-only cells
ref_k_hadamard_after_flatten = ref_head_hadamard_after_flatten
ref_v_hadamard_after_flatten = ref_head_hadamard_after_flatten

## Synthetic paged int4 cache (same slot math as flatten kernel tests)

In [3]:
def build_paged_int4_fixture(
    batch_size: int,
    num_heads: int,
    head_dim: int,
    page_size: int,
    max_seq_len: int,
    cache_seqlens: torch.Tensor,
    dtype: torch.dtype = torch.bfloat16,
):
    """Returns tensors suitable for flatten_kv_cache_sglang(quant_policy=4)."""
    device = cache_seqlens.device
    total_tokens = int(cache_seqlens.sum().item())
    cu_seqlens_k = torch.zeros(batch_size + 1, device=device, dtype=torch.int32)
    cu_seqlens_k[1:] = torch.cumsum(cache_seqlens, dim=0)
    max_pages = (max_seq_len + page_size - 1) // page_size
    page_table = torch.zeros(batch_size, max_pages, device=device, dtype=torch.int32)
    total_slots = batch_size * max_pages * page_size
    cache_loc_list = []
    for b in range(batch_size):
        num_pages = (int(cache_seqlens[b].item()) + page_size - 1) // page_size
        page_table[b, :num_pages] = torch.arange(
            b * max_pages,
            b * max_pages + num_pages,
            device=device,
            dtype=torch.int32,
        )
        seq_start = int(cu_seqlens_k[b].item())
        seq_end = int(cu_seqlens_k[b + 1].item())
        for token_idx in range(seq_start, seq_end):
            token_in_seq = token_idx - seq_start
            page_idx = token_in_seq // page_size
            offset_in_page = token_in_seq % page_size
            page_id = int(page_table[b, page_idx].item())
            slot_id = page_id * page_size + offset_in_page
            cache_loc_list.append(slot_id)
    cache_loc = torch.tensor(cache_loc_list, device=device, dtype=torch.int32)

    h_store = head_dim // 2
    k_buf = torch.zeros(total_slots, num_heads, h_store, device=device, dtype=torch.uint8)
    v_buf = torch.zeros_like(k_buf)
    k_sz = torch.zeros(total_slots, num_heads, 2, device=device, dtype=torch.float32)
    v_sz = torch.zeros_like(k_sz)

    k_orig = torch.randn(total_tokens, num_heads, head_dim, device=device, dtype=dtype)
    v_orig = torch.randn_like(k_orig)
    quantized_set_kv_int4_triton(k_orig, v_orig, cache_loc, k_buf, v_buf, k_sz, v_sz)

    return dict(
        k_buf=k_buf,
        v_buf=v_buf,
        k_sz=k_sz,
        v_sz=v_sz,
        page_table=page_table,
        cache_seqlens=cache_seqlens,
        cu_seqlens_k=cu_seqlens_k,
        total_tokens=total_tokens,
        total_slots=total_slots,
    )

## 1. Sanity: fused K and/or V vs flatten + PyTorch FWHT

In [4]:
def run_sanity(
    order: int = 16,
    rtol: float = 2e-2,
    atol: float = 2e-2,
) -> None:
    batch_size = 2
    num_heads = 8
    head_dim = 128
    page_size = 16
    max_seq_len = 64
    cache_seqlens = torch.tensor([40, 24], device=DEVICE, dtype=torch.int32)
    fx = build_paged_int4_fixture(
        batch_size, num_heads, head_dim, page_size, max_seq_len, cache_seqlens
    )
    common = dict(
        page_size=page_size,
        num_heads=num_heads,
        head_dim_k=head_dim,
        head_dim_v=head_dim,
        quant_policy=4,
        output_dtype=torch.bfloat16,
        max_seq_len_k=max_seq_len,
        out_size=fx["total_tokens"],
    )
    k_base, v_base = flatten_kv_cache_sglang(
        fx["k_buf"],
        fx["v_buf"],
        fx["k_sz"],
        fx["v_sz"],
        fx["page_table"],
        fx["cache_seqlens"],
        fx["cu_seqlens_k"],
        fuse_k_hadamard=False,
        fuse_v_hadamard=False,
        **common,
    )
    k_h_ref = ref_k_hadamard_after_flatten(k_base, order)
    v_h_ref = ref_v_hadamard_after_flatten(v_base, order)

    def call(fk: bool, fv: bool):
        return flatten_kv_cache_sglang(
            fx["k_buf"],
            fx["v_buf"],
            fx["k_sz"],
            fx["v_sz"],
            fx["page_table"],
            fx["cache_seqlens"],
            fx["cu_seqlens_k"],
            fuse_k_hadamard=fk,
            fuse_v_hadamard=fv,
            hadamard_order=order,
            **common,
        )

    kk, vk = call(True, False)
    assert torch.allclose(vk, v_base), "V unchanged when only K fused"
    assert torch.allclose(kk, k_h_ref, rtol=rtol, atol=atol), (kk - k_h_ref).abs().max()
    print("K-only fuse OK", (kk.float() - k_h_ref.float()).abs().max().item())

    kv, vv = call(False, True)
    assert torch.allclose(kv, k_base), "K unchanged when only V fused"
    assert torch.allclose(vv, v_h_ref, rtol=rtol, atol=atol), (vv - v_h_ref).abs().max()
    print("V-only fuse OK", (vv.float() - v_h_ref.float()).abs().max().item())

    kb, vb = call(True, True)
    assert torch.allclose(kb, k_h_ref, rtol=rtol, atol=atol)
    assert torch.allclose(vb, v_h_ref, rtol=rtol, atol=atol)
    print("K+V fuse OK")
    print("sanity OK")


run_sanity()

K-only fuse OK 0.015625
V-only fuse OK 0.015625
K+V fuse OK
sanity OK


## 2. Throughput: baseline vs K fuse vs V fuse vs K+V

Uses `torch.cuda.Event` timing after warmup. **Bytes moved** (same formula for every variant):
`read = slots*(H*D/2)*2 (K,V uint8) + slots*(H*2*4)*2 (scales) + batch*pages*4 (page table)`; `write = total_tokens*H*D*elem*2 (K,V out)`.

In [5]:
def estimate_bytes_moved(
    total_slots: int,
    total_tokens: int,
    batch_size: int,
    max_pages: int,
    num_heads: int,
    head_dim: int,
    elem_bytes: int,
) -> int:
    packed = total_slots * num_heads * (head_dim // 2)
    read_kv = 2 * packed
    read_sz = 2 * (total_slots * num_heads * 2 * 4)
    read_pt = batch_size * max_pages * 4
    write = 2 * total_tokens * num_heads * head_dim * elem_bytes
    return read_kv + read_sz + read_pt + write


def bench_flatten(
    fx: dict,
    common: dict,
    hadamard_order: int,
    fuse_k: bool,
    fuse_v: bool,
    repeats: int = 30,
    warmup: int = 5,
) -> float:
    for _ in range(warmup):
        flatten_kv_cache_sglang(
            fx["k_buf"],
            fx["v_buf"],
            fx["k_sz"],
            fx["v_sz"],
            fx["page_table"],
            fx["cache_seqlens"],
            fx["cu_seqlens_k"],
            fuse_k_hadamard=fuse_k,
            fuse_v_hadamard=fuse_v,
            hadamard_order=hadamard_order,
            **common,
        )
    torch.cuda.synchronize()
    t0 = torch.cuda.Event(enable_timing=True)
    t1 = torch.cuda.Event(enable_timing=True)
    t0.record()
    for _ in range(repeats):
        flatten_kv_cache_sglang(
            fx["k_buf"],
            fx["v_buf"],
            fx["k_sz"],
            fx["v_sz"],
            fx["page_table"],
            fx["cache_seqlens"],
            fx["cu_seqlens_k"],
            fuse_k_hadamard=fuse_k,
            fuse_v_hadamard=fuse_v,
            hadamard_order=hadamard_order,
            **common,
        )
    t1.record()
    torch.cuda.synchronize()
    ms = t0.elapsed_time(t1) / repeats
    return ms


def run_throughput_bench() -> None:
    batch_size = 4
    num_heads = 32
    head_dim = 128
    page_size = 16
    max_seq_len = 2048
    order = 16
    cache_seqlens = torch.full(
        (batch_size,), max_seq_len, device=DEVICE, dtype=torch.int32
    )
    out_dtype = torch.bfloat16
    elem_b = 2

    fx = build_paged_int4_fixture(
        batch_size, num_heads, head_dim, page_size, max_seq_len, cache_seqlens, out_dtype
    )
    max_pages = (max_seq_len + page_size - 1) // page_size
    common = dict(
        page_size=page_size,
        num_heads=num_heads,
        head_dim_k=head_dim,
        head_dim_v=head_dim,
        quant_policy=4,
        output_dtype=out_dtype,
        max_seq_len_k=max_seq_len,
        out_size=fx["total_tokens"],
    )

    nbytes = estimate_bytes_moved(
        fx["total_slots"],
        fx["total_tokens"],
        batch_size,
        max_pages,
        num_heads,
        head_dim,
        elem_b,
    )

    ms0 = bench_flatten(fx, common, order, False, False)
    ms_k = bench_flatten(fx, common, order, True, False)
    ms_v = bench_flatten(fx, common, order, False, True)
    ms_kv = bench_flatten(fx, common, order, True, True)

    def gbps(ms: float) -> float:
        return (nbytes / 1e9) / (ms / 1e3)

    print(f"Approx bytes/call: {nbytes / 1e9:.3f} GB")
    print(
        f"flatten only:     {ms0:.4f} ms  -> {gbps(ms0):.2f} GB/s  (1.00x)"
    )
    print(
        f"+ K FWHT:         {ms_k:.4f} ms  -> {gbps(ms_k):.2f} GB/s  ({ms_k/ms0:.3f}x)"
    )
    print(
        f"+ V FWHT:         {ms_v:.4f} ms  -> {gbps(ms_v):.2f} GB/s  ({ms_v/ms0:.3f}x)"
    )
    print(
        f"+ K+V FWHT:       {ms_kv:.4f} ms  -> {gbps(ms_kv):.2f} GB/s  ({ms_kv/ms0:.3f}x)"
    )


run_throughput_bench()

Approx bytes/call: 0.172 GB
flatten only:     0.0733 ms  -> 2346.94 GB/s  (1.00x)
+ K FWHT:         0.0992 ms  -> 1733.46 GB/s  (1.354x)
+ V FWHT:         0.0965 ms  -> 1781.15 GB/s  (1.318x)
+ K+V FWHT:       0.1612 ms  -> 1066.80 GB/s  (2.200x)


### Optional: int8 + `fuse_k_hadamard` or `fuse_v_hadamard` must raise

In [6]:
try:
    flatten_kv_cache_sglang(
        torch.zeros(1, 1, 1, dtype=torch.uint8, device=DEVICE),
        torch.zeros(1, 1, 1, dtype=torch.uint8, device=DEVICE),
        torch.zeros(1, 1, 2, device=DEVICE),
        torch.zeros(1, 1, 2, device=DEVICE),
        torch.zeros(1, 1, dtype=torch.int32, device=DEVICE),
        torch.tensor([1], device=DEVICE, dtype=torch.int32),
        torch.tensor([0, 1], device=DEVICE, dtype=torch.int32),
        page_size=1,
        num_heads=1,
        head_dim_k=1,
        head_dim_v=1,
        quant_policy=8,
        output_dtype=torch.bfloat16,
        max_seq_len_k=1,
        out_size=1,
        fuse_k_hadamard=True,
    )
except ValueError as e:
    print("expected:", e)
else:
    raise RuntimeError("expected ValueError for int8 + fuse_k_hadamard")

try:
    flatten_kv_cache_sglang(
        torch.zeros(1, 1, 1, dtype=torch.uint8, device=DEVICE),
        torch.zeros(1, 1, 1, dtype=torch.uint8, device=DEVICE),
        torch.zeros(1, 1, 2, device=DEVICE),
        torch.zeros(1, 1, 2, device=DEVICE),
        torch.zeros(1, 1, dtype=torch.int32, device=DEVICE),
        torch.tensor([1], device=DEVICE, dtype=torch.int32),
        torch.tensor([0, 1], device=DEVICE, dtype=torch.int32),
        page_size=1,
        num_heads=1,
        head_dim_k=1,
        head_dim_v=1,
        quant_policy=8,
        output_dtype=torch.bfloat16,
        max_seq_len_k=1,
        out_size=1,
        fuse_v_hadamard=True,
    )
except ValueError as e:
    print("expected v:", e)
else:
    raise RuntimeError("expected ValueError for int8 + fuse_v_hadamard")

expected: fuse_k_hadamard / fuse_v_hadamard are only supported for int4 KV (quant_policy=4)
expected v: fuse_k_hadamard / fuse_v_hadamard are only supported for int4 KV (quant_policy=4)
